# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI(
    api_key=os.environ.get('OPENAI_API_KEY'),
    base_url=os.environ.get('OPENAI_API_BASE_URL', 'https://api.openai.com/v1')
)

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [4]:
 # adjust if your folder layout differs
DATA_DIR = Path("../data")

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', golden['j01'])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'clean â€” clear company + role + years'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [ ]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    return [
        {
            "role": "user",
            "content": f"""
Extract the company, role, years_experience_required (integer or null) 
and notes (additional relevant information or observations) information from this job posting:

Return only valid JSON.

Job posting:
{snippet_text}
"""
        }
    ]



def prompt_few_shot(snippet_text: str) -> list[dict]:
    return [
        {
            "role": "user",
            "content": f"""
Extract company, role, minimum experience, and notes.

Example 1:
Input: Acme Corp is hiring a Senior Engineer requiring 5+ years.

Output:
{{
    "company": "Acme Corp",
    "role": "Senior Engineer",
    "years_experience_required": 5,
    "notes": "Requires 5+ years of experience."
}}

Example 2:
Input: TechNova seeks a Python Developer with at least 3 years.

Output:
{{
    "company": "TechNova",
    "role": "Python Developer",
    "years_experience_required": 3,
    "notes": "Requires at least 3 years of experience."
}}

Example 3:
Input: DataWorks is hiring a Data Analyst.

Output:
{{
    "company": "DataWorks",
    "role": "Data Analyst",
    "years_experience_required": null,
    "notes": "Experience requirement not specified."
}}

Now extract the information:

Job posting:
{snippet_text}

Return only valid JSON with the same four fields.
"""
        }
    ]



def prompt_structured(snippet_text: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": """
You are an expert recruiter and job description analyst.

Extract these four fields:

{
    "company": "string or null",
    "role": "string or null",
    "years_experience_required": "integer or null",
    "notes": "string"
}

Rules:
- Extract the hiring company.
- Extract the actual job title.
- Extract minimum required experience.
- Use null if information is unavailable.
- Do not invent missing information.
- Use notes for additional relevant observations.
- Return only valid JSON.
"""
        },
        {
            "role": "user",
            "content": f"Analyze this job posting:\n{snippet_text}"
        }
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    return [
        {
            "role": "user",
            "content": f"""
Analyze this job posting carefully.

Identify:
1. Company
2. Job role
3. Minimum required years of experience
4. Additional relevant observations (notes)

Consider ambiguous wording and distinguish required
experience from preferred experience.

Think through the extraction step by step internally.

Return only valid JSON:

{{
    "company": "string or null",
    "role": "string or null",
    "years_experience_required": "integer or null",
    "notes": "string"
}}

Job posting:
{snippet_text}
"""
        }
    ]


STRATEGIES = {
    'zero_shot': 'prompt_zero_shot',
    'few_shot': 'prompt_few_shot',
    'structured': 'prompt_structured',
    'cot': 'prompt_cot',
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [6]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    # TODO: extract + parse the JSON, return a dict or None 

    text = text.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

    
async def run_one(strategy_name: str, snippet_dict: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""

    #snippet_text = snippet_dict["snippet"]
    if strategy_name=='zero_shot':
            question = prompt_zero_shot(snippet_text = snippet_dict["snippet"])
    elif strategy_name=='few_shot':
            question = prompt_few_shot(snippet_text = snippet_dict["snippet"])
    elif strategy_name=='structured':
            question = prompt_structured(snippet_text = snippet_dict["snippet"])
    elif strategy_name=='cot':
            question = prompt_cot(snippet_text = snippet_dict["snippet"])

    # TODO: call the model, time the call, compute cost, parse the response
    
    # start time
    mstart = time.time() 

    # call the model
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=question,
        temperature=TEMPERATURE
    )
    # raw response
    raw_response = response.choices[0].message.content
    # parsed response
    parsed_response = parse_response(raw_response)
    # latency
    mtime = time.time() - mstart  
    # Input tokens
    input_tokens = response.usage.prompt_tokens
    # Output tokens
    output_tokens = response.usage.completion_tokens
    # Input cost  
    input_cost = input_tokens * (0.15 / 1000000)
    # Output cost
    output_cost = output_tokens * (2.50 / 1000000)
    # Total cost
    total_cost = input_cost + output_cost 
        
    
    return {
            "strategy": strategy_name,
            "snippet_id": snippet_dict["id"],
            "raw_response": response.choices[0].message.content,
            "parsed_response": parsed_response,
            "latency_s": mtime,
            "cost_usd": total_cost
        }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    # TODO: build the task list, gather, return results
    task_list= []
    for strategy_num in STRATEGIES:
        for i in range(len(snippets)):
            task = run_one(strategy_num, snippets[i])
            task_list.append(task)

    results = await asyncio.gather(*task_list)
    return results

In [7]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')


Got 40 results.


## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [ ]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    # TODO: count exact matches (with normalisation)

    accuracy_count = 0
    if extracted is None:
        return 0
    for field in ['company', 'role', 'years_experience_required']:
        if field in extracted and field in gold:
            if isinstance(extracted[field], str) and isinstance(gold[field], str):
                if extracted[field].strip().lower() == gold[field].strip().lower():
                    accuracy_count += 1
            else:
                if extracted[field] == gold[field]:
                    accuracy_count += 1
    return accuracy_count
        


def parse_success(extracted: dict | None):
    """Return True if the extracted dict is valid and has all three fields."""
    parse_count = False
    if extracted is None:
        parse_count = False
    elif all(field in extracted for field in ['company', 'role', 'years_experience_required']):
        parse_count = True
    return parse_count



async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    # TODO: prompt the judge with both the gold and the extracted, ask for a 1-4 score

    RUBRIC = """
    fields = ['company', 'role', 'years_experience_required']
    score 1-25, where 25 is the best score and 1 is the worst score by :
    1. All three fields correct
    2. Two of three correct, no fabricated data
    3. One of three correct, or fabricated a field
    4. None correct or unparsable
    Return only a JSON object with a single integer field 'score'.
    """


    llm_judge_response = await client.chat.completions.create(
            model=JUDGE_MODEL,
            temperature=0.0,
            messages=[
                {"role": "system", "content":
                    f"You are a strict evaluator of LLM answers.\n\n{RUBRIC}\n\n"
                    "Return your scores as a JSON object with a single integer field 'score'."},
                {"role": "user", "content":
                    f"Question context for parsing the job snippet:\n{snippet_text}\n\n"
                    f"Golden answer for parsed job snippet: {gold}\n\n"
                    f"Candidate answer to evaluate: {extracted}\n\n"},
            ],
            response_format = {"type": "json_object"}
        )

    return json.loads(llm_judge_response.choices[0].message.content)  

In [7]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row

scored = []

llm_list = []
# llm judge score
for i in range(len(results)):
    llm_score = score_llm_judge(
            snippet_text=snippets[i % 10],
            extracted=results[i]['parsed_response'],
            gold=golden
        )
    llm_list.append(llm_score)

llm_scores = await asyncio.gather(*llm_list)


for i in range(len(results)):
    # make a copy to avoid modifying the original
    result_dict = results[i].copy() 

    extracted = result_dict['parsed_response']
    gold = golden[result_dict['snippet_id']]

    # accuracy score
    accuracy = score_accuracy(extracted, gold)
    result_dict['accuracy'] = accuracy

    # parse success score
    parse_suc = parse_success(extracted)
    result_dict['parse_success'] = parse_suc


    # llm judge score attachment
    result_dict['llm_judge_score'] = llm_scores[i]['score']
    

    scored.append(result_dict)

## Step 5 — Build the comparison table

In [8]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary['parse_success'] = summary['parse_success'] * 100

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.8,100.0,25.0,0.002,3.023
few_shot,2.8,100.0,25.0,0.002,2.878
structured,2.8,100.0,25.0,0.002,3.031
zero_shot,2.6,100.0,20.5,0.002,2.758


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```